# K-Means++ Initialization from Scratch — ANSWER KEY

## Objective
Implement the K-Means++ initialization strategy from scratch using NumPy, then use it to seed K-Means clustering.

This is the instructor solution copy — every **YOUR CODE HERE** cell from the assignment is filled in below.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 2. Generate the Dataset

In [ ]:
cluster_1 = np.random.randn(20, 2) * 0.6 + np.array([1, 1])
cluster_2 = np.random.randn(20, 2) * 0.6 + np.array([6, 6])
cluster_3 = np.random.randn(20, 2) * 0.6 + np.array([1, 7])

X = np.vstack([cluster_1, cluster_2, cluster_3])

print("X shape:", X.shape)

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], s=40)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Raw Data")
plt.show()

## 3. Core K-Means Functions (Provided)

In [ ]:
def compute_distance(point, centroid):
    return np.linalg.norm(point - centroid, ord=2)

def find_closest_centroid(point, centroids):
    distances = np.linalg.norm(centroids - point, axis=1)
    return np.argmin(distances)

def assign_clusters(X, centroids):
    idx = []
    for point in X:
        idx.append(find_closest_centroid(point, centroids))
    return np.array(idx)

def compute_centroids(X, idx, K):
    new_centroids = []
    for k in range(K):
        points = X[idx == k]
        new_centroids.append(np.mean(points, axis=0))
    return np.array(new_centroids)

def compute_cost(X, idx, centroids):
    m = len(X)
    total = 0
    for i in range(m):
        centroid = centroids[idx[i]]
        total += np.sum((X[i] - centroid)**2)
    return total / m

def run_kmeans(X, initial_centroids, max_iters=100):
    centroids = initial_centroids.copy()
    for i in range(max_iters):
        idx = assign_clusters(X, centroids)
        new_centroids = compute_centroids(X, idx, len(centroids))
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids
    return centroids, idx

## 4. Why K-Means++?

Plain K-Means picks its initial centroids uniformly at random, which can lead to poor local optima. K-Means++ spreads the initial centroids out on purpose, weighting new picks toward points that are far from the centroids already chosen.

## 5. Step 1 — Distance to the Nearest Chosen Centroid

Given as the worked example.

In [ ]:
def nearest_centroid_distance(point, centroids):
    distances = [compute_distance(point, c) for c in centroids]
    return min(distances)

In [ ]:
test_centroids = [np.array([1.0, 1.0]), np.array([6.0, 6.0])]
print(nearest_centroid_distance(np.array([1.5, 1.5]), test_centroids))

## 6. Step 2 — Squared Distances for All Points

### Solution
Loop over every point in `X`, compute its distance to the nearest chosen centroid, and square each result.

In [ ]:
def compute_squared_distances(X, centroids):
    squared_distances = np.array([
        nearest_centroid_distance(point, centroids) for point in X
    ]) ** 2

    return squared_distances

In [ ]:
test_sq = compute_squared_distances(X, [X[0]])
print(test_sq[:5])
print(test_sq.shape)

## 7. Step 3 — Weighted Random Choice of the Next Centroid

### Solution
Normalize the squared distances into probabilities, then sample one index according to that distribution.

In [ ]:
def choose_next_centroid(X, squared_distances):
    probabilities = squared_distances / np.sum(squared_distances)
    index = np.random.choice(len(X), p=probabilities)

    return X[index]

## 8. Step 4 — Complete K-Means++ Initialization

### Solution
Start with one random centroid, then repeatedly pick the next one weighted by squared distance to the nearest already-chosen centroid, until there are `K` centroids.

In [ ]:
def kmeans_plus_plus_init(X, K):
    centroids = [X[np.random.randint(len(X))]]

    for _ in range(1, K):
        squared_distances = compute_squared_distances(X, centroids)
        next_centroid = choose_next_centroid(X, squared_distances)
        centroids.append(next_centroid)

    return np.array(centroids)

In [ ]:
initial_centroids_pp = kmeans_plus_plus_init(X, K=3)
print(initial_centroids_pp)

## 9. Step 5 — Run K-Means Using K-Means++ Initialization

### Solution

In [ ]:
final_centroids_pp, idx_pp = run_kmeans(X, initial_centroids_pp)

plt.figure(figsize=(7,5))

plt.scatter(X[:, 0], X[:, 1], c=idx_pp, s=60, cmap='Blues')
plt.scatter(final_centroids_pp[:, 0], final_centroids_pp[:, 1], marker='X', color='red', s=200)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("K-Means++ Clustering Result")
plt.show()

## 10. Step 6 — Compare K-Means++ vs. Random Initialization

### Solution

In [ ]:
n_trials = 20
K = 3

random_costs = []
pp_costs = []

for _ in range(n_trials):
    random_indices = np.random.choice(len(X), K, replace=False)
    initial_random = X[random_indices]
    centroids_r, idx_r = run_kmeans(X, initial_random)
    random_costs.append(compute_cost(X, idx_r, centroids_r))

    initial_pp = kmeans_plus_plus_init(X, K)
    centroids_p, idx_p = run_kmeans(X, initial_pp)
    pp_costs.append(compute_cost(X, idx_p, centroids_p))

print("Random init  -> mean cost:", np.mean(random_costs), " std:", np.std(random_costs))
print("K-Means++    -> mean cost:", np.mean(pp_costs),     " std:", np.std(pp_costs))

## 11. Step 7 — Elbow Method Using K-Means++

### Solution

In [ ]:
costs_pp = []

for K in range(2, 6):
    initial_pp = kmeans_plus_plus_init(X, K)
    centroids_p, idx_p = run_kmeans(X, initial_pp)
    cost = compute_cost(X, idx_p, centroids_p)
    costs_pp.append(cost)

In [ ]:
plt.plot(range(2, 6), costs_pp, marker='o')
plt.xlabel("K")
plt.ylabel("Cost")
plt.title("Elbow Method (K-Means++ Initialization)")
plt.show()

# Test your understanding — Why Squared Distance?

### Solution

**Answer:** Squaring the distance sharpens the contrast between near and far points — a point twice as far away becomes **four times** as likely to be picked, not just twice. If unsquared `D(x)` were used instead, the probability distribution over points would be flatter (closer to uniform), so the algorithm would behave more like plain random initialization and lose much of its tendency to spread centroids apart. K-Means++ specifically uses `D(x)^2` because that choice is what gives the algorithm its provable approximation guarantee (expected cost within `O(log K)` of the optimal clustering) — it isn't an arbitrary choice.

# Test your understanding — Stability

### Solution

**Answer:** `pp_costs` typically has both a **lower mean** and a **lower standard deviation** than `random_costs`. The lower mean means K-Means++ finds better clusterings on average — it avoids the very bad outcomes that happen when random initialization happens to place two centroids close together. The lower standard deviation means it's also more *consistent*: run after run lands near the same good result, whereas plain random initialization occasionally produces a much worse clustering purely by bad luck in the starting positions. In other words, K-Means++ trades a small amount of extra computation at initialization time for materially more reliable convergence.